# Solution: Introduction to PyTorch - Application

Welcome!

In this notebook you'll get to practice the basics of PyTorch. We'll implement a simple neural network to predict the bicycle traffic flow in sectors of New York.

<center><figure>

  <img src="https://live.staticflickr.com/5630/20875991748_e47eb7e6fb_b.jpg" width=600/>
<figcaption>
<sub><sup>  </sup></sub></figcaption>

</figure></center>

**<u>Content:</u>**

* Introduction
* Data visualization
* Data pre-processing
* Linear model
* Training loop
* Model prediction on test dataset

## Preliminaries

In [ ]:
# We import some basic libraries for some operations later in the notebook
import os
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float32)
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from cycler import cycler
import seaborn as sns

from urllib.request import urlretrieve

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

In case you want to explore the GPU capabilities for training a model, upload this notebook on [Google Colab](https://colab.research.google.com/drive/).
In Google Colab, select the option "Upload notebook" in the menu "File".

For enabling the GPU, you can go to the menu "Edit", then "Notebook settings", and then change the hardware accelerator to GPU.

You can automatically set the GPU as prefered device if there is one available. Run the following cell to set the device to GPU (cuda) or CPU (cpu). We will use this variable `device` to move the PyTorch tensors to the respective device.

Both data and model should be in the same device. We'll point out the places where the device variable enters into play.


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device: ", device)

## Introduction

Traffic prediction is a critical aspect of urban planning and management. The ability to forecast traffic flow not only enables efficient traffic control but also helps prevent potential disasters resulting from unexpected traffic spikes.

The primary objective of traffic prediction is to forecast traffic volume, this is inflow and outflow of vehicles, for various regions within a city at a specific future time. These predictions are based on historical traffic observations. As stated by Ji et al. (2023), the accuracy of these predictions is vital for effective traffic management and safety.

In this Jupyter notebook, we will use PyTorch to predict bicycle traffic in the city of New York.

<sup id="f1">J. Ji, J. Wang, C. Huang, et al. "Spatio-Temporal Self-Supervised Learning for Traffic Flow Prediction". in AAAI 2023.</sup>  

### Dataset

For our analysis, we will utilize the *NYCBike1* dataset<sup id="a1">*</sup>, which records bike rentals in New York City on an hourly basis. This dataset covers the period from April 1st to September 30th in 2014.

The dataset consists of 128 disjoint geographical sectors. These samples are further divided into training, validation, and test sets with a ratio of 7:1:2. This division allow training and evaluating our traffic prediction model effectively.





<sup id="f1">*Zhang et al., "Deep spatio-temporal residual networks for citywide crowd flows prediction." Proceedings of the AAAI conference on artificial intelligence. Vol. 31. No. 1. 2017.</sup>  

Let's load the dataset and take a first look at it. Execute the following cell to load the dataset and print the sizes of the files

#### Training data

In [ ]:
# Download the dataset (if necessary)
url = "https://surfdrive.surf.nl/s/GarWpNzMTJGN6pL/download"
filename = "train.npz"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

data_path = f"train.npz"
data_training = np.load(data_path, allow_pickle=True)
for file in data_training.files:
    print(file, data_training[file].shape)

Here we can observe that the training dataset has 4 different variables. We will focus on the `x` and `y` variables in the dataset.

The `x` variable represents the input data. The `y` variable represents target values that the model will try to predict.

Both variables have 4 dimensions: (N, T, S, F)
being `N` the number of samples, `T` the number of time steps, `S` the number of sectors, and `F` the type of flow (`0` = inflow; `1` = outflow).

For example, the input `data_training['x'][0, 1, 50, 1]` represents the data in the first example (`N=0`) of the second hour (`T=1`) for the 50th city sector (`S=50`) and the outflow (`F=1`).

In [ ]:
data_training["x"][0, 1, 50, 1]

#### Validation data

In [ ]:
# Download the dataset (if necessary)
url = "https://surfdrive.surf.nl/s/Ae5K7k3B4daaoSe/download"
filename = "val.npz"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)


data_path = f"val.npz"
data_val = np.load(data_path, allow_pickle=True)
for file in data_val.files:
    print(file, data_val[file].shape)

### Approach

We will develop a neural network that learns from historical traffic data and predicts future traffic volumes for a geographical sector in New York City.

In the following sections, we will visualize and pre-process the data. Next, we'll develop a model with the PyTorch framework. By the end of this notebook, you should have a better understanding of how PyTorch can be used to solve a real-world problem in the field of urban traffic prediction.

## Data visualization

We can visualize the inflow and outflow of bicycles for a city sector in of the training examples. First, let's define the example we want to visualize.

In [ ]:
example_number = 224  # Number between 0 and 3022

Now, let's extract the information from the selected example:

In [ ]:
example_x = data_training["x"][example_number]
example_y = data_training["y"][example_number]

In [ ]:
print(example_x.shape)
print(example_y.shape)

From the training example, we see that the inputs have 19 hours of data for each of the 128 city sectors. The target values have 1 hour of data (the following hour after the input 19 hours) in all of the 128 city sectors.

In the following code, we can extract the information of the city sector and visualize the inflow and outflow of bicycles.

First, we select a city sector and then we extract the information of the inflow and outflow of bicycles.

In [ ]:
city_sector = 100  # Number between 0 and 127

We create all the axes for the plot

In [ ]:
traffic_inflow_x = example_x[:, city_sector, 0]
traffic_outflow_x = example_x[:, city_sector, 1]

traffic_inflow_y = example_y[:, city_sector, 0]
traffic_outflow_y = example_y[:, city_sector, 1]

time_axis_x = np.arange(0, len(traffic_inflow_x))
time_axis_y = [time_axis_x[-1] + len(traffic_inflow_y)]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.yaxis.get_major_locator().set_params(integer=True)
ax.xaxis.get_major_locator().set_params(integer=True)

# Plot the data points
ax.plot(time_axis_x, traffic_inflow_x, "rs-", markersize=5, label="Traffic inflow")
ax.plot(time_axis_x, traffic_outflow_x, "bo-", markersize=5, label="Traffic outflow")

ax.plot(time_axis_y, traffic_inflow_y, "rs", markersize=10, label="Target inflow")
ax.plot(time_axis_y, traffic_outflow_y, "bo", markersize=10, label="Target outflow")

# Add a title and axis labels
ax.set_title("Example of traffic timeseries", fontsize=16)
ax.set_xlabel("Hour", fontsize=14)
ax.set_ylabel("Number of bicycles", fontsize=14)

# Add a legend
ax.legend(fontsize=14)

# Add a grid
ax.grid(True)

# Show the plot
plt.show()

**Exercise - Visualizing city sectors:**

By modifying the above code, answer the following questions:
1. Is the bike traffic different for different sectors of the city? For answering this question, try sectors 1, 10, and 100 for the example number 224.

<!-- solution -->
The bike traffic is different for different sectors of the city. For the example 224, sector 1 has no traffic at all. Sector 10 has a higher inflow and outflow of bicycles, and sector 100 fluctuates and then it shows a sudden decrease; nevertheless, the target values for this sector are not zero.
<!-- solution -->

## Selection of City Sector

From now on, we will work with an individual city sector. Select one, this could be for instance the same sector you chose before for visualization.

In [ ]:
sector_ID = (
    city_sector  # enter ID of sector we want to use for developing our PyTorch model
)

We extract the information of the selected city sector from the training dataset.

In [ ]:
training_x = data_training["x"][:, :, sector_ID, :]
training_y = data_training["y"][:, :, sector_ID, :]

## Data preprocessing

First, we create normalizer classes to help us pre-process the data. Normalization is often crucial for training neural networks as it ensures consistent scaling of input data, enabling the model to learn and converge more effectively. One way to normalize the data is what we call *min-max normalization*. This normalization scales the data to a fixed range, usually between 0 and 1. The normalization is performed by subtracting the minimum value of the data and dividing by the range.

In [ ]:
class MinMaxNormalizer:
    def __init__(self, data_training):
        self.max = data_training.max()
        self.min = data_training.min()

    def normalize(self, data):
        return (data - self.min) / (self.max - self.min)

    def denormalize(self, data):
        return data * (self.max - self.min) + self.min

Other popular form of normalization is Stardarization, in which the data is scaled to have a mean of zero and a standard deviation of one. In the following exercise, you can implement your own standardization normalizer.

**Exercise: Implementing a Data Normalizer Class in Python**

In this exercise, you are provided with a Python class named `StandardNormalizer`. This class is designed to normalize and denormalize data based on the mean and standard deviation of a training dataset.

Your task is to use this class to perform the following operations:

1. Implement a method to normalize a new dataset using the mean and standard deviation of the training dataset. 
2. Implement a method to denormalize the normalized data back to its original form.

Answer:
1. For this case, it is possible to calculate the mean and standard deviation of the entire dataset instead of normalizing each feature separately. Why is that?

The `StandardNormalizer` class has the following methods:

- `__init__(self, data_training)`: This method initializes the `StandardNormalizer` class with a training dataset. It calculates the mean and standard deviation of the training dataset.

- `normalize(self, data)`: This method takes a new dataset as input and normalizes it using the mean and standard deviation of the training dataset.

- `denormalize(self, data)`: This method takes a normalized dataset as input and denormalizes it back to its original form using the mean and standard deviation of the training dataset.


Answer:
<!-- solution -->
In this case, it is possible to only extract an average value and a standard deviation because all of the features are the same variable: Bike traffic rentals.  
<!-- solution -->

In [ ]:
class StandardNormalizer:
    def __init__(self, data_training):
        # ---------------------- student exercise --------------------------------- #
        self.means = data_training.mean()
        self.std_devs = data_training.std()
        # ---------------------- student exercise --------------------------------- #

    def normalize(self, data):
        # ---------------------- student exercise --------------------------------- #
        return (data - self.means) / self.std_devs
        # ---------------------- student exercise --------------------------------- #

    def denormalize(self, data):
        # ---------------------- student exercise --------------------------------- #
        return data * self.std_devs + self.means
        # ---------------------- student exercise --------------------------------- #

The `StandardNormalizer` class above implements standardization, where data is scaled based on the mean and standard deviation of the training data. The `MinMaxNormalizer` class, instead scales data to a specified range (usually 0 to 1) based on the minimum and maximum values in the training data, ensuring uniform scaling of features. Both methods are widely used. Choose one and continue with the exercise.

Let's create the normalizer and normalize the data

In [ ]:
data_normalizer = StandardNormalizer(training_x)

In [ ]:
normalized_training_x = data_normalizer.normalize(training_x)
normalized_training_y = data_normalizer.normalize(training_y)

**Exercise:**

Normalize the validation data using the normalizer created with the training data.

In [ ]:
# normalized_val_x = ...
# normalized_val_y = ...
# ---------------------- student exercise --------------------------------- #
normalized_val_x = data_normalizer.normalize(data_val["x"][:, :, sector_ID, :])
normalized_val_y = data_normalizer.normalize(data_val["y"][:, :, sector_ID, :])
# ---------------------- student exercise --------------------------------- #

## PyTorch Dataset

Now, we need to create a custom dataset class to store the `x` and `y` examples in one object.

Notice: We specify in the constructor (`__init__`) that our dataset will be loaded to the current device; this is CPU or GPU.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data_x, data_y):
        self.x = (
            torch.from_numpy(data_x).float().to(device)
        )  # Here the data is converted from numpy arrays to torch tensors in the respective device.
        self.y = (
            torch.from_numpy(data_y).float().to(device)
        )  # Here the data is converted from numpy arrays to torch tensors in the respective device.

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

The `CustomDataset` class defined above extends PyTorch's `Dataset` class to handle our specific dataset. In this implementation, the input numpy arrays `data_x` and `data_y` are converted to PyTorch tensors and stored as `self.x` and `self.y`. The `__len__` method returns the total number of samples in the dataset, and `__getitem__` fetches a specific sample (both input x and output y) based on an index `idx`; `__getitem__` will essentially supply the data to the neural network for training, validation and testing.

This structure of custom datasets in PyTorch is very helpful, offering flexibility to handle various data types and structures. PyTorch provides multiple methods to load data, each suited for different purposes and data formats, enhancing the versatility of data handling in deep learning projects.

In [ ]:
training_set = CustomDataset(normalized_training_x, normalized_training_y)
validation_set = CustomDataset(normalized_val_x, normalized_val_y)

Now that we have normalized dataset, we can create and train a fully connected neural network.

## Fully connected Neural Network (FNN) model

**Exercise: Implementing a Fully connected Neural Network (FNN) in PyTorch**

In this exercise, you are tasked with implementing a simple feedforward neural network (FNN) using PyTorch. This model consists of a series of linear operations followed by non-linear activation functions. The inputs are processed by flattening them into a single dimension, thus ignoring the spatial and temporal information of the bikes dataset. (There are other models that we will explore later on that consider these inductive biases.)

The FNN should have the following specifications:

1. The FNN should have minimum three layers: an input layer, a hidden layer, and an output layer. The number of neurons in the input and output layers should be specified by the parameters `input_size` and `output_size` respectively. The hidden layer should have 64 neurons by default, but this should be customizable via an optional parameter `hidden_features`.

2. Each layer should be a linear layer, implemented using `nn.Linear`. This means that the output of each layer is a linear function of its input.

3. The activation function for the first two layers should be ReLU (Rectified Linear Unit), implemented using `nn.ReLU`. Be careful not to use a ReLU activation function for the output layer in case you normalized using standarization. Can you explain why?

4. The `forward` method should take a batch of input tensors, flatten them into 2D tensors, pass them through the layers of the network, and return the output tensors.

Your task is to define a class `FNN` that implements this feedforward neural network. The class should have an `__init__` method to initialize the layers and a `forward` method to compute the output of the network. 
> Hint: You should use the `FNN` implementation of the introductory notebook on PyTorch as a blueprint.

<!-- solution -->
The question about the activation function for the output layer is related to the normalization method. If we use ReLU, the output of the network will be a value equal or larger than zero. However, the standarized training values have negative values (as it was designed to have zero mean).
<!-- solution -->

In [ ]:
class FNN(nn.Module):
    # We create FNN as a subclass of nn.Module
    # In this way we import some important methods from the parent class
    def __init__(self, input_size, output_size, hidden_features=64):
        super().__init__()
        # In the init method we specify the layers of the neural network
        # ---------------------- student exercise --------------------------------- #
        self.lin1 = nn.Linear(input_size, hidden_features)
        self.lin2 = nn.Linear(hidden_features, hidden_features)
        self.lin3 = nn.Linear(hidden_features, output_size)
        # ---------------------- student exercise --------------------------------- #

    def forward(self, x):
        # The FNN takes a flatten input of dimensions [N, F],
        # where N are the samples and F are the input features (input_size)

        x = x.reshape(x.shape[0], -1)

        # ---------------------- student exercise --------------------------------- #
        x = self.lin1(x)
        x = nn.ReLU()(x)
        x = self.lin2(x)
        x = nn.ReLU()(x)
        x = self.lin3(x)
        # ---------------------- student exercise --------------------------------- #

        return x

### Model instantiation
After creating the FNN class, we must define an instance of it with the parameters that fit our application.

In [ ]:
input_size = 19 * 2  # 19 hours, 2 time-series;
output_size = 2  # 2 individual outputs;

print(f"The size of out input will be = {input_size}")
print(f"The size of out output will be = {output_size}")

model = FNN(input_size, output_size, hidden_features=32)
print(model)

As soon as the model is created, it is stored in the CPU. We can run the following cell to load the model to the device that we are using.

In [ ]:
model = model.to(device)

### Dataloaders instantiation

 The PyTorch `DataLoader` automates the process of loading batches of data from a dataset. It efficiently handles tasks like batching (grouping data into batches of a specified size), and shuffling (randomizing the order of data for the training set to improve learning). For the training loader (`train_loader`), data is shuffled to introduce randomness and prevent overfitting, while for the validation loader (`validation_loader`), shuffling is typically disabled to evaluate the model consistently on the same data order.

In [ ]:
# Create the training and validation dataloaders to "feed" data to the model in batches
batch_size = 64
train_loader = DataLoader(training_set, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=batch_size, shuffle=False)

These variables carry the information of 3023 training examples and 432 validation examples in batches of size 64 (or the value that you choose). Each examples has 19 hours as input features. The target values have 1 hour of data (the following hour after the input 19 hours) for a selected city sector.

## Training loop

We now define for convenience some function that execute a training loop for 1 epoch and evaluate the performance of a given model.
In our experiments, we use Mean Average Error (MAE) as training and evaluation metric.

**Exercise: Implementing a Training Loop in PyTorch**

In this exercise, you are tasked with implementing a training loop for a PyTorch model. The training loop should perform the following steps for each batch of data:

1. Use the model to make predictions based on the inputs. The syntax for this is `model(inputs)`. (Note that the model is callable, just like a function.)
2. Compute the loss between the predictions and the target output using a loss function.
3. Append the loss to a list of losses for later analysis.
4. Use backpropagation to compute the gradients of the loss with respect to the model's parameters.
5. Use an optimizer to update the model's parameters based on the computed gradients.
6. Reset the gradients to zero for the next iteration.

Your task is to write a Python loop that iterates over all batches provided by `loader` and performs the steps outlined above.
> Hint: You should use the training loop shown in the introductory notebook on PyTorch as a blueprint. The main difference is that we are iterating through all the batches provided by the loader to perform a single epoch of training.

In [ ]:
def train_epoch(model, loader, optimizer, loss_function):
    model.train()  # specifies that the model is in training mode
    losses = []

    for batch in loader:
        inputs, target = batch

        # 1. Model prediction
        # ---------------------- student exercise --------------------------------- #
        preds = model(inputs)
        # ---------------------- student exercise --------------------------------- #

        # 2. Loss function. Tip: Use `reshape(target.shape[0], -1)` to give the target tensor the right dimensions.
        # ---------------------- student exercise --------------------------------- #
        loss = loss_function(preds, target.reshape(target.shape[0], -1))
        # ---------------------- student exercise --------------------------------- #

        # 3. Append the loss to a list of losses for later analysis. Hint: Use `detach()` and `item()` to extract the loss value from the loss tensor.
        losses.append(loss.detach().item())

        # 4. Backpropagate. Hint: Use the method .backward() to compute the gradients.
        # ---------------------- student exercise --------------------------------- #
        loss.backward()  # compute the gradients using backpropagation
        # ---------------------- student exercise --------------------------------- #

        # 5. Optimize the weights with the optimizer Hint: Use the method .step() to update the weights.
        # ---------------------- student exercise --------------------------------- #
        optimizer.step()  # update the weights with the optimizer
        # ---------------------- student exercise --------------------------------- #

        # 6. Reset the computed gradients. Hint: Use the method .zero_grad() to reset the gradients.
        # ---------------------- student exercise --------------------------------- #
        optimizer.zero_grad(set_to_none=True)  # reset the computed gradients
        # ---------------------- student exercise --------------------------------- #

    return np.array(losses).mean()

**Exercise: Implementing an Evaluation Loop in PyTorch**

In this exercise, you are tasked with implementing an evaluation loop for a PyTorch model. The evaluation loop should perform the following steps for each batch of data:

1. Extract the inputs and the target output from the batch.
2. Use the model to make predictions based on the inputs.
3. Compute the loss between the predictions and the target output using a loss function.
4. Append the loss to a list of losses for later analysis.

Your task is to write a Python loop that iterates over all batches provided by `loader` and performs the steps outlined above.

Use the previous exercise as guide. What is the main difference respect to the training loop?

<!-- solution -->
The main difference is that we don't update the model parameters. We only compute the loss and append it to a list.
<!-- solution -->

In [ ]:
def evaluation(model, loader, loss_function):
    model.eval()  # specifies that the model is in evaluation mode
    losses = []

    # Remove gradients computations since we are only evaluating and not training
    with torch.no_grad():
        for batch in loader:
            # extract inputs x and outputs y from batch
            inputs, target = batch
            # ---------------------- student exercise --------------------------------- #

            # Model prediction
            preds = model(inputs)

            # Loss function
            loss = loss_function(preds, target.reshape(target.shape[0], -1))
            losses.append(loss.item())
            # ---------------------- student exercise --------------------------------- #

    return np.array(losses).mean()

Now, we set the hyperparameters for the training process.

In [ ]:
# Set training parameters
learning_rate = 0.0001
num_epochs = 200

# Create the optimizer to train the neural network via back-propagation
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Create loss function
loss_function = nn.L1Loss()  # MAE Loss

In the following cell, we train the model.

In [ ]:
# Now we can use the functions we just created to train our model
for epoch in range(1, num_epochs + 1):
    # Model training
    train_loss = train_epoch(model, train_loader, optimizer, loss_function)
    validation_loss = evaluation(model, validation_loader, loss_function)

    # print loss every 10 epochs
    if epoch % 10 == 0:
        print(
            "epoch:",
            epoch,
            "\t training loss:",
            np.round(train_loss, 4),
            "\t validation loss:",
            np.round(validation_loss, 4),
        )

## Model prediction on test dataset

Now that our model is trained, we can use it to infer the traffic flow in the test dataset.

We are going to repeat the process of processing and normalization for one example of the test dataset.
1. We load the test dataset.
2. We normalize the test dataset using the normalizer created with the training dataset.
3. We create a custom dataset to store the `x` and `y` examples in one object.

In [ ]:
# Download the dataset (if necessary)
url = "https://surfdrive.surf.nl/s/tYbmAcEYXKyLEm8/download"
filename = "test.npz"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

# 1. Load the test dataset
data_path = f"test.npz"
data_test = np.load(data_path, allow_pickle=True)

for file in data_test.files:
    print(file, data_test[file].shape)

Now, we can evaluate our model in one of the test examples.

In [ ]:
example_number = 17  # Maximum 863

In [ ]:
test_x = torch.tensor(
    data_test["x"][example_number, :, sector_ID, :], dtype=torch.float32
).to(device)
test_y = torch.tensor(
    data_test["y"][example_number, :, sector_ID, :], dtype=torch.float32
).to(device)

normalized_test_x = data_normalizer.normalize(test_x)
normalized_test_y = data_normalizer.normalize(test_y)

**Exercise: Denormalizing Model Predictions**

Your task is to use the model to make a prediction on the test sample and then denormalize the prediction to bring it back to the original scale. 

The steps are as follows:

1. Reshape the test sample to have the correct input shape (1,38).
2. Use the model to make a prediction on the reshaped test sample.
3. Detach the prediction from the computation graph to prevent further gradient computations.
4. Use the `denormalize()` method of `data_normalizer` to denormalize the prediction.


In [ ]:
# ---------------------- student exercise --------------------------------- #

# 1. Reshape
normalized_test_x_reshaped = normalized_test_x.reshape(1, -1)

# 2. Use the model
normalized_prediction = model(normalized_test_x_reshaped)

# 3. Detach predictions
detached_prediction = normalized_prediction.detach()

# 4. Denormalize predictions
predictions = data_normalizer.denormalize(detached_prediction)

# ---------------------- student exercise --------------------------------- #

For visualization, we need to send all tensors back to CPU. We do it with the method `.to('cpu')`

In [ ]:
# ------------------------------------------------- #
traffic_inflow_x = test_x[:, 0].to("cpu")
traffic_outflow_x = test_x[:, 1].to("cpu")

traffic_inflow_y = test_y[:, 0].to("cpu")
traffic_outflow_y = test_y[:, 1].to("cpu")

pred_traffic_inflow_y = predictions[:, 0].to("cpu")
pred_traffic_outflow_y = predictions[:, 1].to("cpu")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.yaxis.get_major_locator().set_params(integer=True)
ax.xaxis.get_major_locator().set_params(integer=True)

# Plot the data points
ax.plot(time_axis_x, traffic_inflow_x, "rs-", markersize=5, label="Traffic inflow")
ax.plot(time_axis_x, traffic_outflow_x, "bo-", markersize=5, label="Traffic outflow")

ax.plot(time_axis_y, traffic_inflow_y, "rs", markersize=10, label="Target inflow")
ax.plot(time_axis_y, traffic_outflow_y, "bo", markersize=10, label="Target outflow")

ax.plot(
    time_axis_y, pred_traffic_inflow_y, "rx", markersize=10, label="Predicted inflow"
)
ax.plot(
    time_axis_y, pred_traffic_outflow_y, "b+", markersize=10, label="Predicted outflow"
)

# Add a title and axis labels
ax.set_title("Example of traffic timeseries", fontsize=16)
ax.set_xlabel("Hour", fontsize=14)
ax.set_ylabel("Number of bicycles", fontsize=14)

# Add a legend
ax.legend(fontsize=14)

# Add a grid
ax.grid(True)

# Show the plot
plt.show()

We see that the model is able to predict the inflow and outflow of bicycles in the city sector with reasonable accuracy for some examples. The model is able to capture the general trend of the data, but it is not able to predict the exact values. For now, this is good enough for us. 

In this tutorial, we used PyTorch to estimate how many bikes travel through the streets of New York. We used a simple fully connected neural network. However, this is just the beginning! As we advance in the course, we will explore more models and techniques to improve the accuracy of our predictions. 